# multi_collinearity

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/multi_collinearity.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先用 cap program drop 清除可能已存在的同名程序，避免重复定义报错。

In [ ]:
%%stata
set seed 19880505
cap program drop dgp

定义模拟程序 dgp：语法选项 obs 控制样本量（默认 20），b 控制真实系数（默认 0），mc=1 时制造多重共线性。tempvar 声明临时变量，先设定观测数。

当 mc=1 时，让 x2 = (3/√10)·x1 + (1/√10)·ε，这样 x1 与 x2 的相关系数高达 0.9，制造严重的多重共线性；mc=0 时 x2 独立生成，作为对照组。

生成被解释变量 y = b·x1 + x2 + u，然后对 y 做 OLS 回归，返回 x1 的系数估计与标准误。

计算 t 统计量并与 t(n−3) 分布的 0.975 分位数比较：若绝对值超过临界值则记 rejected=1（拒绝原假设），否则为 0。同时记录系数估计是否为负（neg），用于观察共线性下估计符号混乱的现象。

In [ ]:
%%stata
program define dgp, rclass
	syntax [, obs(integer 20) b(real 0) mc(real 1)]
	drop _all
	set obs `obs'
	tempvar x1 x2 y sigma
	gen `x1'=rnormal()
	if `mc'==1{
		gen `x2'=3/sqrt(10)*`x1'+1/sqrt(10)*rnormal()
	}
	else {
		gen `x2'=rnormal()
	}
	gen `y'=`b'*`x1'+`x2'+rnormal()
	quietly: reg `y' `x1' `x2'
	return scalar b=_b[`x1']
	return scalar se=_se[`x1']
	if abs(_b[`x1']/_se[`x1'])>=invt(`obs'-3,0.975) {
		return scalar rejected=1
	}
	else {
		return scalar rejected=0
	}
	return scalar neg=_b[`x1']<0
end

第一组模拟：真实系数 b=0（原假设成立），在多重共线性下重复 2000 次。su 显示拒绝频率即检验的实际显著性水平（size），应接近名义水平 5%，说明共线性本身不扭曲检验水平，但会增大标准误。

In [ ]:
%%stata
// multicolinearty, size
simulate rejected=r(rejected) b=r(b) se=r(se) neg=r(neg), reps(2000):dgp
su

第二组模拟：真实系数 b=1（备择假设成立），在多重共线性下考察检验功效（power）。由于共线性使标准误膨胀，拒绝频率（功效）会明显降低。

In [ ]:
%%stata
// multicolinearty, power
simulate rejected=r(rejected) b=r(b) se=r(se) neg=r(neg), reps(2000):dgp, b(1)
su

第三组模拟：去掉多重共线性（mc=0），同样 b=1。对比上一组可以看到，没有共线性时功效明显更高，说明多重共线性的危害主要体现在降低检验功效、增大系数估计的不确定性。

In [ ]:
%%stata
// no multicolinearty, power
simulate rejected=r(rejected) b=r(b) se=r(se) neg=r(neg), reps(2000):dgp, b(1) mc(0)
su

第四组模拟：在多重共线性下把样本量从 20 提高到 100。可以看到，增大样本量能有效缓解共线性带来的功效损失——多重共线性本质上是小样本问题，随着样本增加其影响会减弱。

In [ ]:
%%stata
// multicolinearty, power with large N
simulate rejected=r(rejected) b=r(b) se=r(se) neg=r(neg), reps(2000):dgp, b(1) obs(100)
su